In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os

print("ZA JSE Web Scraping Tool v.1.0")

# Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'ZA JSE SQL Ready {}.xlsx'.format(str(now).replace(":", ".")[:-7])
writer = ExcelWriter(filename)

# Assigning the folders that are going to be used in the process
scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder, 'tempfolder')
os.chdir(scriptfolder)
outputfolder = r'D:\Regulators\output\ready'

# Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
    for temp_file in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, temp_file))
else:
    os.mkdir(tempfolder)

# Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
         "download.prompt_for_download": False,
         "download.default_directory": tempfolder}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()

# Creating dictionary with Regcodes and their respective URLs
regdict = {
    'ZA JSE 1': 'Equity Issuer',
    'ZA JSE 2': 'ETF Issuer',
    'ZA JSE 3': 'Warrant Issuer',
    'ZA JSE 4': 'Interest Rate Issuer',
    'ZA JSE 5': 'Debt Issuer',
    'ZA JSE 6': 'Structured Product Issuer',
    'ZA JSE 7': 'ETN Issuer',
    'ZA JSE 8': 'Asset Backed Securities (ABS) Issuer',
    'ZA JSE 9': 'Hybrid Issuer',
    'ZA JSE 10': 'Actively Managed Certificate Issuer',
    'ZA JSE 11': 'Actively Managed ETF Issuer'
    }

# Creating dictionary to containg regulators data and then be converted to a pandas' DataFrame
sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],
           'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
           'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [],
           'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [],
           'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [],
           'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')
main_url = 'https://clientportal.jse.co.za/companies-and-financial-instruments'
for reg, (listname) in regdict.items():
    print('Working with {}'.format(reg))
    driver.get(main_url)
    sleep(4)

    wait = WebDriverWait(driver, 15)
    try:
        filter_selector = f'li.filterType[data-issuertype="{listname}"]'
        filter_element = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, filter_selector)))

        try:
            prev_fist = driver.find_element(By.CSS_SELECTOR, 'div.companyName').text.sptrip()
        except:
            prev_first = ''

        filter_element.click()
        sleep(1)

        wait.until(lambda d: d.find_element(By.CSS_SELECTOR, 'div.companyName').text.strip() != prev_first)
        companies = driver.find_elements(By.CSS_SELECTOR, 'div.companyName')

    except Exception as e:
        print(f"filter not found for {listname}, skipping. Error: {e}")
        continue

    suspended_count = 0  # Counter for suspended companies

    for i in range(len(companies)):
        try:
            companies = driver.find_elements(By.CSS_SELECTOR, 'div.companyName')
            company = companies[i]
            try:
                status_div = companies[i].find_element(By.XPATH,
                                                       './/following-sibling::div[contains(@class, "status")]')
                if 'suspended' in status_div.text.lower():
                    print(f"Skipping suspended company: {companies[i].text.strip()}")
                    suspended_count += 1
                    continue
            except:
                pass

            company.click()
            sleep(2)

            try:
                name = driver.find_element(By.CSS_SELECTOR, 'div.companyProfileName').text.strip()
                name = name.split(':', 1)[-1].strip()
            except:
                name = ''
            try:
                id_type = driver.find_element(By.CSS_SELECTOR, 'h4.companyRegHeader').text.strip()
            except:
                id_type = ''
            try:
                internal_id = driver.find_element(By.CSS_SELECTOR, 'h4.companyRegData').text.strip()
            except:
                internal_id = ''

            phone, email, website, fax = '', '', '', ''
            try:
                contacts = driver.find_elements(By.CSS_SELECTOR, 'div.companyContact ul li')
                for li in contacts:
                    txt = li.text.lower()
                    try:
                        a_tag = li.find_element(By.TAG_NAME, 'a')
                        value = a_tag.text.strip()
                        if 'tel' in txt:
                            phone = value
                        elif 'fax' in txt:
                            fax = value
                        elif 'email' in txt:
                            email = value
                        elif 'http' in a_tag.get_attribute('href').lower():
                            website = value
                    except:
                        continue
            except:
                pass

            try:
                address_container = driver.find_element(By.CSS_SELECTOR, 'div.companyContactAddressHeader')
                address_divs = address_container.find_elements(By.CSS_SELECTOR, 'div')
                address = ' '.join(div.text.strip() for div in address_divs if div.text.strip())
            except:
                address = ''

            try:
                address2_container = driver.find_element(By.CSS_SELECTOR, 'div.companyContactPostalHeader')
                address2_divs = address2_container.find_elements(By.CSS_SELECTOR, 'div')
                address2 = ' '.join(div.text.strip() for div in address2_divs if div.text.strip())
            except:
                address2 = ''

            sqldict['Name'].append(name)
            sqldict['InternalID_1'].append(internal_id)
            if internal_id:
                sqldict['InternalID_1_type'].append(id_type)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Phone'].append(phone)
            sqldict['Email'].append(email)
            sqldict['Fax'].append(fax)
            sqldict['Website'].append(website)
            sqldict['Address_1'].append(address)
            sqldict['Address_2'].append(address2)
            sqldict['ListName'].append(listname)
            sqldict['RegCtry'].append('ZA')
            sqldict['RegCode'].append('JSE')
            sqldict['ListCode'].append(reg.split(' ')[-1])
            sqldict['RegulationType'].append("Regulated")
            max_len = max(len(v) for v in sqldict.values())
            for k in sqldict:
                while len(sqldict[k]) < max_len:
                    sqldict[k].append('')
            driver.back()
            sleep(2)

            current_url = driver.current_url
            if "data:" in current_url or "about:blank" in current_url or len(driver.find_elements(By.CSS_SELECTOR, 'li.filterType')) == 0:
                print("Blank or invalid page detected after back(). Reloading main URL...")
                driver.get(main_url)
                sleep(3)

            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'li.filterType')))
            sleep(2)

            for attempt in range(5):
                try:
                    filter_selector = f'li.filterType[data-issuertype="{listname}"]'
                    filter_element = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, filter_selector)))
                    filter_element.click()
                    wait.until(
                        lambda d: d.find_element(By.CSS_SELECTOR, 'li.filterType.filterTypeActive').get_attribute('data-issuertype').replace("'", "").replace('"', "").strip().lower()
                        == listname.replace("'", "").replace('"', "").strip().lower()
                    )
                    break
                except Exception as e:
                    print(f"Retry {attempt+1} failed to click filter: {e}")
                    sleep(2)
            else:
                print(f"Failed to click filter for {listname} after several attempts, skipping...")
                continue

            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.companyName')))
            sleep(1)
            companies = driver.find_elements(By.CSS_SELECTOR, 'div.companyName')

        except Exception as e:
            print(f"Error on company index {i}: {e}")

    print(f"Suspended companies in '{listname}': {suspended_count}")

os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

# Remove duplicates based on Name, InternalID_1, InternalID_1_type, and ListCode
df = df.drop_duplicates(subset=['Name', 'InternalID_1', 'InternalID_1_type', 'ListCode'], keep='first')

df.to_excel(writer, 'SQL Ready', index=False)

# writer.save()
writer.close()
driver.quit()
    
    
    